# Scraping data from a wikipedia page (pandas) 

In [3]:
from bs4 import BeautifulSoup
import requests

### https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue

#### first lets import the soup. (note: because modern websites block bots to protect their servers, you must identify yourself in your header to look like a human.

In [9]:
url = 'https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

page = requests.get(url, headers=headers)
print(page.status_code)

soup = BeautifulSoup(page.text, 'html')

200


In [11]:
#print(soup)

##### the first thing we need to do is specify what data were looking for. 
##### from inspecting the page we can find the class for the table we want 
##### There are two tables on this page both with the same class but this should be fine

In [63]:
#soup.find_all('table')[0]  #Here weve used indexing to get at the table we want. this works because its just a list.

#class="wikitable sortable" #but you could have used the class

In [16]:
table = soup.find_all('table')[0]

##### The first thing were going to try to get are the titles. From inspection all the titles have th tags (note: were doing the find_all on the table not the whole soup)

In [49]:
world_titles=table.find_all('th')

In [50]:
world_titles

[<th id="mwPw">Rank</th>,
 <th id="mwQA">Name</th>,
 <th id="mwQQ">Industry</th>,
 <th id="mwQg">Revenue <br id="mwQw"/>(USD millions)</th>,
 <th id="mwRA">Revenue growth</th>,
 <th id="mwRQ">Employees</th>,
 <th id="mwRg">Headquarters</th>]

##### now we can loop throught this list and extract just the title names

In [51]:
world_table_titles = [title.text.strip() for title in world_titles]

In [52]:
print(world_table_titles)

['Rank', 'Name', 'Industry', 'Revenue (USD millions)', 'Revenue growth', 'Employees', 'Headquarters']


### now we can start building out the database using pandas

In [53]:
import pandas as pd

In [55]:
df = pd.DataFrame(columns=world_table_titles)
df

,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters


### now we need to start pulling in the data itself
#### the tr tag represents the rows and the d tag represents the data

In [58]:
column_data=table.find_all('tr') #this returns all the table data as a list 

##### now we can loop through the list of data weve created to extract the information as text similarly to what we did with the column titles.
##### then insert each row into the data frame created above.

In [61]:
for row in column_data[1:]: #note we start at position 1 because there is an empty entry at position 0
    row_data=row.find_all('td')
    individual_row_data = [data.text.strip() for data in row_data] #now weve extracted the row of data as text. The next step is to put it in the data frame
    #now appending it into the dataframe
    length=len(df)
    df.loc[length] = individual_row_data #a better explaination of this is in the pandas series.
    

In [62]:
df

,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters
0,1,Walmart,Retail,"680,985",5.1%,"2,100,000","Bentonville, Arkansas"
1,2,Amazon,Retail and cloud computing,"637,959",11.0%,"1,556,000","Seattle, Washington"
2,3,UnitedHealth Group,Healthcare,"400,278",7.7%,"400,000","Minnetonka, Minnesota"
3,4,Apple,Technology,"391,035",2.0%,"164,000","Cupertino, California"
4,5,CVS Health,Healthcare,"372,809",4.2%,"259,500","Woonsocket, Rhode Island"
...,...,...,...,...,...,...,...
95,96,General Dynamics,Aerospace and defense,"47,716",12.9%,"117,000","Reston, Virginia"
96,97,Coca-Cola,Beverage,"47,061",2.9%,"69,700","Atlanta, Georgia"
97,98,TIAA,Financials,"46,946",2.6%,"15,623","New York City, New York"
98,99,Travelers,Insurance,"46,423",12.2%,"34,000","New York City, New York"


In [ ]:
#df.to_csv(r'\Companies.csv', index= False) #insert a file path to export as a csv 